In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Setup & Data Simulation
# MAGIC Este notebook se encarga de preparar el entorno **Landing (Raw)**.
# MAGIC 
# MAGIC **Funcionalidades:**
# MAGIC 1. **RESET_AND_LOAD:** Borra los datos en el Volumen de destino y vuelve a copiar los originales (Ideal para reiniciar pruebas).
# MAGIC 2. **APPEND_ONLY:** Copia datos nuevos sin borrar los existentes (Simula llegada de datos diarios).

# COMMAND ----------

import os
import shutil

# --- 1. CONFIGURACION DE WIDGETS (INTERFAZ) ---
# Esto crea un menu desplegable arriba para elegir el modo
dbutils.widgets.dropdown("modo_ejecucion", "RESET_AND_LOAD", ["RESET_AND_LOAD", "APPEND_ONLY"], "Modo de Ejecucion")

mode = dbutils.widgets.get("modo_ejecucion")
print(f"Modo seleccionado: {mode}")

# COMMAND ----------

# --- 2. CONFIGURACION DE RUTAS ---
current_user = spark.sql("SELECT current_user()").first()[0]

# RUTAS ORIGEN (Workspace / Git - Donde tienes los materiales originales)
# Ajusta esta ruta si tu estructura de carpetas es diferente en el Workspace
base_workspace = f"/Workspace/Users/{current_user}/preparacion_databricks_data_engineer/materiales"
src_drivers = f"{base_workspace}/inbox_drivers"
src_sensors = f"{base_workspace}/inbox_sensors"

# RUTAS DESTINO (Volumen - Donde DLT leera los datos)
base_volume = "/Volumes/ecomotive/landing/ecomotive_vol"
dst_drivers = f"{base_volume}/inbox_drivers"
dst_sensors = f"{base_volume}/inbox_sensors"

print(f"Origen: {base_workspace}")
print(f"Destino (Volumen): {base_volume}")

# COMMAND ----------

# --- 3. FUNCIONES DE UTILIDAD ---

def reset_landing_zone(path):
    """Borra el contenido del destino para empezar de cero."""
    if os.path.exists(path):
        print(f"Eliminando datos antiguos en: {path}")
        shutil.rmtree(path)
    else:
        print(f"La ruta no existia (esta limpia): {path}")

def copy_data(src, dst):
    """Copia datos de origen a destino."""
    try:
        if not os.path.exists(src):
            print(f"ERROR: La ruta origen no existe: {src}")
            return

        print(f"Copiando desde: {src}")
        print(f"   -> Hacia: {dst}")
        # dirs_exist_ok=True permite copiar sobre una carpeta existente (util para APPEND)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Copia completada con exito.")
        
    except Exception as e:
        print(f"Error al copiar: {e}")

# COMMAND ----------

# --- 4. EJECUCION DEL FLUJO ---

# A) Si el modo es RESET, limpiamos primero
if mode == "RESET_AND_LOAD":
    print("--- INICIANDO LIMPIEZA TOTAL ---")
    reset_landing_zone(dst_drivers)
    reset_landing_zone(dst_sensors)
    print("--- LIMPIEZA COMPLETADA ---\n")

# B) Copia de datos (Se ejecuta en ambos modos)
print("--- INICIANDO INGESTA DE DATOS ---")

# Simulamos llegada de Drivers
copy_data(src_drivers, dst_drivers)

# Simulamos llegada de Sensores
copy_data(src_sensors, dst_sensors)

# COMMAND ----------

# --- 5. VERIFICACION FINAL ---
print(f"\n--- ESTADO ACTUAL DEL VOLUMEN ({mode}) ---")

print("Conductores en Inbox:")
try:
    display(dbutils.fs.ls(dst_drivers))
except:
    print("Carpeta vacia o no existente.")

print("Sensores en Inbox:")
try:
    display(dbutils.fs.ls(dst_sensors))
except:
    print("Carpeta vacia o no existente.")